# E1.9 · Model and agent lifecycle governance

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

---

**Risk.** Re-indexing treated as maintenance, not change.

**Control.** Retraining, fine-tuning and re-indexing as change-management events.

**This lab.** Write the gate a re-index has to pass.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E1.9"))

Model and agent lifecycle governance. The lifecycle events that matter are the ones with no ticket.

In [ ]:
from cybercommons import planes, grc, soc
import time

EVENTS = {
 "new agent deployed":        ("ticketed", "caught by existing process"),
 "tool added to manifest":    ("no ticket", "changes blast radius silently"),
 "prompt edited in console":  ("no ticket", "changes behaviour, not code"),
 "provider upgrades model":   ("no ticket", "you may not even be told"),
 "scope widened in IAM":      ("sometimes", "depends on your IAM review"),
 "agent decommissioned":      ("rarely",   "identity often outlives the agent"),
}
for e, (state, why) in EVENTS.items():
    print(f"{e:28s}{state:12s}{why}")

Two of these are detectable with things already built in this curriculum: the manifest diff (A1.1) and drift monitoring (D1.7).

In [ ]:
W = planes.Tool
d = planes.diff_manifests(
    planes.Manifest("a", [W("read_file")], rung="L2"),
    planes.Manifest("a", [W("read_file"),
                          W("deploy_prod", writes=True, scope="org",
                            reversible=False)], rung="L2"))
print("manifest change detected:", d["added"], f"blast +{d['delta']}")

now = time.time()
base = soc.Baseline({"read_file": 1.0}, actions_per_hour=100)
print("behaviour change detected:",
      base.compare([soc.Event(now, "a", "deploy")] * 10)["verdict"])

The decommissioning row is the one people miss: a retired agent whose identity still exists is a standing credential with no owner.

### Expect

The lifecycle table shows four of six events untracked. The manifest diff detects the added tool with its blast-radius delta, and the baseline comparison reports significant drift.

### Your turn

Query your identity provider for non-human identities with no authentication in 90 days. Each one is a decommissioning that never finished.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E1.9.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*